In [1]:
library(Seurat)
library(nicheDE)
library(qs)
library(glue)
library(ggplot2)
library(patchwork)
library(latex2exp)
library(scCustomize)
library(scico)
library(ComplexHeatmap)
library(ggsci)
library(latex2exp)
library(ggrepel)
library(org.Hs.eg.db)
library(clusterProfiler)
library(RColorBrewer)
library(Polychrome)
library(stringr) 
library(parallel)
library(ggpubr)
library(dplyr)
library(tidyr)

Warning message:
“package ‘Seurat’ was built under R version 4.4.3”
Loading required package: SeuratObject

Warning message:
“package ‘SeuratObject’ was built under R version 4.4.3”
Loading required package: sp

Warning message:
“package ‘sp’ was built under R version 4.4.2”

Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t


Warning message:
“package ‘qs’ was built under R version 4.4.3”
qs 0.27.3. Announcement: https://github.com/qsbase/qs/issues/103

Warning message:
“package ‘ggplot2’ was built under R version 4.4.3”
Warning message:
“package ‘patchwork’ was built under R version 4.4.3”
Warning message:
“package ‘latex2exp’ was built under R version 4.4.3”
scCustomize v3.0.1
If you find the scCustomize useful please cite.
See 'samuel-marsh.github.io/scCustomize/articles/FAQ.html' for citation info.

Warning message:
“package ‘ComplexHeatmap’ was built under R version 4.4.2”
Loading required package: grid

ComplexHeatmap ver

In [2]:
colors_all <- c(
  "#DA80DA", "#815481", "#C040C0", "#E1AFE1", "#3F0034",
  "#EDABB9", "#EB5C79", "#A06A75", "#C00028",
  "#EB675E", "#A23E36",
  "#540F54", "#53407F",
  "#DFA38A", "#8C3612", "#623623", "#916350", "#DAC3C3",
  "#F8770B", "#E09E3A", "#CD7225", "#FFC990", "#AC5812",
  "#FEE083", "#897538", "#E7B419", "#BCA048",
  "#6F8BE2", "#3053BC",
  "#6D9F58", "#9EB766", "#BDCB10", "#3A6527", "#9EA743",
  "#E2E8A7", "#5A6209", "#8FE36B",
  "#818A31",
  "#9FC5E8", "#23D9F1",
  "#64C6A6",
  "#AAAAAA"
)
ct_cols <- c("M2"=colors_all[10],
             "M5"=colors_all[28],
             "F2"=colors_all[30],
             "F7"=colors_all[24],
             "M1"=colors_all[4],
             "NK"=colors_all[39],
             "M10"=colors_all[17],
             "M3"=colors_all[32],
             "F3"=colors_all[29],
             "F4"=colors_all[12],
             "B3"=colors_all[41],
             "E0"=colors_all[19],
             "B1"=colors_all[7],
             "B2"=colors_all[11])

In [3]:
spatialVariablePlot <- function(obj, plot_cts, focal_ct=NULL, focal_size=0.85){
    coords <- GetTissueCoordinates(obj)
    rownames(coords) <- coords$cell
    ap_ratio <- diff(range(coords[, "x"]))/diff(range(coords[, "y"]))
    scale_bar_x_start <- min(coords[, "y"]) + 50
    scale_bar_x_end <- min(coords[, "y"]) + 50 + 1000
    scale_bar_y_pos <- min(coords[, "x"] + 50)
    coords |> dplyr::select(-cell) -> coords
    cells_of_interest <- which(Idents(obj) %in% plot_cts)
    coords$celltype <- "Background"
    coords$celltype[cells_of_interest] <- as.character(Idents(obj)[cells_of_interest])
    coords$celltype <- factor(coords$celltype, levels=c("Background", sort(unique(as.character(Idents(obj)[cells_of_interest])))))
    plot_order <- seq_len(nlevels(coords$celltype))
    names(plot_order) <- levels(coords$celltype)
    coords$plot_order <- plot_order[as.character(coords$celltype)]
    coords %>%
    arrange(plot_order) -> coords
    alpha_vals <- rep(1, nlevels(coords$celltype))
    ct_palalpha_vals <- rep(1, nlevels(coords$celltype))
    names(alpha_vals) <- levels(coords$celltype)
    alpha_vals["Background"] <- 0.2
    size_vals <- rep(0.4, nlevels(coords$celltype))
    names(size_vals) <- levels(coords$celltype)
    if(length(focal_ct) > 0){
       size_vals[focal_ct] <- focal_size 
    }
    ggplot(coords, aes(y, x, fill=celltype, alpha=celltype, size=celltype)) +
                    geom_point(stroke=0.05, shape=21, color="black") +
                    scale_fill_manual(values=ct_cols,
                                      breaks=setdiff(levels(coords$celltype), "Background")) +
                    scale_alpha_manual(values=alpha_vals,
                                       breaks=setdiff(levels(coords$celltype), "Background"),
                                       name="") +
                    scale_size_manual(values=size_vals,
                                      breaks=setdiff(levels(coords$celltype), "Background"),
                                      name="") +
                    theme_void() +
                    theme(legend.key.size=unit(0.3, 'cm'),
                          aspect.ratio=ap_ratio,
                          legend.text=element_text(size=4),
                          plot.title = element_text(size=6)) +
                    labs(color="", fill="", size="") +
                    guides(fill = guide_legend(override.aes=list(size=2, alpha=1))) +
                    annotate("segment",
                             x=scale_bar_x_start,
                             xend=scale_bar_x_end,
                             y=scale_bar_y_pos,
                             yend=scale_bar_y_pos,
                             linewidth=0.5) +
                    annotate("text",
                             x=(scale_bar_x_start+scale_bar_x_end)/2,
                             y=scale_bar_y_pos,
                             label=paste0(1, " mm"),
                             vjust=1.5,
                             size=1.5)
}

In [4]:
all_ra_samples <- list.dirs("/data1/deyk/harry/RA_Xenium/data/post_qc_data_baysor/", recursive=FALSE, full.names=TRUE)
all_sample_ids <- str_extract(all_ra_samples, "RA\\d+[A-Z]*")
height_width_param <- c("6&8", "6.5&7", "6&8", "5&9", "6&8", "7.5&4", "6&10", "6&10", "8&10", "6&8")
names(height_width_param) <- c("RA401", "RA331", "RA442", "RA443", "RA457", "RA480", "RA494", "RA519", "RA362", "RA489")

# Figure 2A

In [5]:
plot_M2_M5_F7 <- lapply(all_sample_ids, function(sample){
    figure_height_width <- as.numeric(str_split(height_width_param[sample], "&")[[1]])
    obj <- qread(glue("/data1/deyk/harry/RA_Xenium/data/post_qc_data_baysor/{sample}/final_version_seurat_obj.qs"))
    plot.tmp <- spatialVariablePlot(obj, plot_cts=c("M2", "M5", "F7"), focal_ct="F7")
    ggsave(plot=plot.tmp,
           filename=glue("/data1/deyk/harry/RA_Xenium/results/figures/manuscript_final_version_figures/figure2/2A/{sample}.png"),
           height=figure_height_width[1],
           width=figure_height_width[2],
           dpi=500,
           limitsize=FALSE,
           bg="white")
})

In [5]:
# plot_M2_F2_F7 <- lapply(all_sample_ids, function(sample){
#     figure_height_width <- as.numeric(str_split(height_width_param[sample], "&")[[1]])
#     obj <- qread(glue("/data1/deyk/harry/RA_Xenium/data/post_qc_data_baysor/{sample}/final_version_seurat_obj.qs"))
#     m2 <- spatialVariablePlot(obj, "M2")
#     m5 <- spatialVariablePlot(obj, "M5")
#     m2 + m5 -> plots.tmp
#     ggsave(plot=plots.tmp,
#            filename=glue("/data1/deyk/harry/RA_Xenium/results/figures/manuscript_final_version_figures/figure2/2A/{sample}.png"),
#            height=figure_height_width[1],
#            width=figure_height_width[2]*3,
#            dpi=500,
#            limitsize=FALSE,
#            bg="white")
# })